In [ ]:
%pip install -q pandas 

# Load data

In [ ]:
from json import loads
import pandas as pd

DATA_FILE = "ai_news_2024-06-26.json"

with open(DATA_FILE) as f:
    data_json = loads(f.read())

df = pd.DataFrame(data_json.values())
df.head()

# Embedding

In [ ]:
%pip install -q openai

### Combine title + body as `text`

In [ ]:
df["text"] = df["title"] + " - " + df["body"]

df.head()

In [ ]:
## TODO : load embeddings from mongodb instead.

# 2D Vis

In [ ]:
%pip install -q scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib

### Re load csv file

In [ ]:
df = pd.read_csv("ai_news_2024-06-26_embedded.csv")

In [ ]:
import plotly.express as px
import pandas as pd

# Convert the date column to datetime if it's not already
df["date"] = pd.to_datetime(df["date"])

# Filter for articles from 2024
df_2024 = df[df["date"].dt.year == 2024]
# Extract just the date part (without time) and count articles per day
daily_counts = df_2024["date"].dt.date.value_counts().sort_index()

# Convert the index to datetime for better x-axis formatting
daily_counts.index = pd.to_datetime(daily_counts.index)

# Create the bar chart
fig = px.bar(
    x=daily_counts.index,
    y=daily_counts.values,
    labels={"x": "Date", "y": "Number of Articles"},
    title="Number of Articles per Day in 2024",
)

# Customize the layout
fig.update_layout(
    xaxis_tickformat="%Y-%m-%d",
    xaxis_title="Date",
    yaxis_title="Number of Articles",
    bargap=0.1,
)

# Show the plot
fig.show()


In [ ]:
matrix = np.array(df.ada_embedding.apply(eval).to_list())


# Create a t-SNE model and transform the data
tsne = TSNE(
    n_components=2,
    perplexity=15,
    random_state=42,
    init="random",
    learning_rate=150,
)
vis_dims = tsne.fit_transform(matrix)  # type:ignore

In [ ]:
%pip install -q plotly nbformat

In [ ]:
import plotly.express as px
import textwrap


def wrap_text(text, width=50):
    return "<br>".join(textwrap.wrap(text, width=width))


# Create a new DataFrame with the t-SNE results, titles, and wrapped bodies
tsne_df = pd.DataFrame(vis_dims, columns=["tsne_1", "tsne_2"])
tsne_df["title"] = df["title"]
tsne_df["wrapped_body"] = df["body"].apply(wrap_text)

# Create an interactive scatter plot using Plotly
fig = px.scatter(
    tsne_df,
    x="tsne_1",
    y="tsne_2",
    hover_data=["title", "wrapped_body"],
    title="2D t-SNE projection of news articles",
    labels={"tsne_1": "t-SNE feature 1", "tsne_2": "t-SNE feature 2"},
)

# Update the hover template to show both title and wrapped body
fig.update_traces(
    hovertemplate="<b>Title:</b> %{customdata[0]}<br><br><b>Body:</b> %{customdata[1]}"
)

# Adjust the hover mode to show the full text
fig.update_layout(hoverdistance=100, hovermode="closest")

# Show the plot
fig.show()

### Clustering

In [ ]:
%pip install -qU hdbscan langchain langchain_community langchain_core langchain_anthropic 

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain.prompts import PromptTemplate
from langchain.schema import StrOutputParser
from textwrap import dedent



def generate_cluster_title(article_titles: list[str]) -> str:
    llm = ChatAnthropic(model_name="claude-3-haiku-20240307")  # type:ignore

    prompt = dedent("""Given the following news article titles, generate a single 3-4 words title that summarizes what the cluster of articles is about:

                                <titles>
                                {titles}
                                </titles>

                                Summary title:""")


    prompt_template = PromptTemplate.from_template(prompt)

    chain = prompt_template | llm | StrOutputParser()

    titles = "\n".join(f"- {title}" for title in article_titles[:5])

    return chain.invoke({"titles":titles})



generate_cluster_title(
    ['Etched is building an AI chip that only runs one type of model', "Transformer model chipmaker Etched.ai raises $120M to challenge Nvidia's market dominance", 'AI chip startup Etched raises $120 million to expand supply', 'Etched raises $120M in challenge to Nvidia in AI with transformer chips', 'NVIDIA-Backed Astrocade Raises $12M For A Promising AI Game Creation Platform']
)

In [ ]:
import plotly.express as px
import textwrap
import hdbscan
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import euclidean_distances


def wrap_text(text, width=50):
    return "<br>".join(textwrap.wrap(text, width=width))


def get_cluster_center(points):
    return np.mean(points, axis=0)


def get_closest_points(points, center, n=5):
    distances = euclidean_distances([center], points)[0]  # type: ignore
    closest_indices = np.argsort(distances)[:n]
    return closest_indices


# Assuming vis_dims is your 2D t-SNE data
# Apply HDBSCAN clustering
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5)
cluster_labels = clusterer.fit_predict(vis_dims)

# Create a new DataFrame with the t-SNE results, titles, wrapped bodies, and cluster labels
tsne_df = pd.DataFrame(vis_dims, columns=["tsne_1", "tsne_2"])
tsne_df["title"] = df["title"]
tsne_df["wrapped_body"] = df["body"].apply(wrap_text)
tsne_df["cluster"] = cluster_labels

# Group points and titles by cluster
cluster_points = defaultdict(list)
cluster_titles = defaultdict(list)
for cluster, point, title in zip(tsne_df["cluster"], vis_dims, tsne_df["title"]):
    cluster_points[cluster].append(point)
    cluster_titles[cluster].append(title)

# Generate titles for each cluster using the 5 closest articles to the center
cluster_names = {-1: "Noise"}  # Keep 'Noise' for cluster -1
for cluster, points in cluster_points.items():
    if cluster != -1:
        center = get_cluster_center(points)
        closest_indices = get_closest_points(points, center)
        closest_titles = [cluster_titles[cluster][i] for i in closest_indices]
        cluster_names[cluster] = generate_cluster_title(closest_titles)

# Apply the new cluster names
tsne_df["cluster_name"] = tsne_df["cluster"].map(cluster_names)

# Update the 'cluster' column with the new names
tsne_df["cluster"] = tsne_df["cluster_name"]

# Drop the temporary 'cluster_name' column
tsne_df = tsne_df.drop("cluster_name", axis=1)

In [ ]:
# ... (previous code remains the same)

# Create an interactive scatter plot using Plotly
fig = px.scatter(
    tsne_df,
    x='tsne_1',
    y='tsne_2',
    color='cluster',
    hover_data=['title', 'wrapped_body', 'cluster'],  # Add 'cluster' to hover_data
    title="2D t-SNE projection of news articles with HDBSCAN clustering",
    labels={'tsne_1': 't-SNE feature 1', 'tsne_2': 't-SNE feature 2'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)

# Update the hover template to show title, wrapped body, and cluster name
fig.update_traces(
    hovertemplate='<b>Title:</b> %{customdata[0]}<br><br>'
                  '<b>Body:</b> %{customdata[1]}<br><br>'
                  '<b>Cluster:</b> %{customdata[2]}'  # Use customdata[2] for cluster name
)

# Adjust the hover mode to show the full text
fig.update_layout(hoverdistance=100, hovermode='closest')

# Show the plot
fig.show()


In [ ]:
# show number of clusters

n_clusters = len(tsne_df["cluster"].unique()) - 1  # Exclude 'Noise' cluster

print(f"Number of clusters: {n_clusters}")

In [ ]:
tsne_df["cluster"].unique().tolist()

In [ ]:
from pprint import pprint

noise_articles = tsne_df[tsne_df["cluster"] == "Noise"]

print(f"Number of articles in 'Noise' cluster: {len(noise_articles)}")

pprint(noise_articles.title.tolist()[:100])

### TODO : 
-  reduce dimensions to 256 and try again clustering
- know more about openai's embedding feature that allows us to trim embeddings without loosing meaning
- automatize daily fetching of news
- get older news from the past and insert them in db